# Práctica 1 · De SMILES a coordenadas 3D
## Grafos moleculares, incrustación ETKDG y optimización GFN2-xTB

> **Bloque 1 — Generación de estructuras**  ·  Manual de Química Computacional UNAM

| | |
|:--|:--|
| **Molécula semilla** | Cafeína — `Cn1cnc2c1c(=O)n(c(=O)n2C)C` |
| **Herramientas** | RDKit · xtb · py3Dmol |
| **Tiempo estimado** | 45 min (semilla) + 2 h (bosque + análisis) |
| **Requisitos previos** | Hibridación, geometría TRPEV, fórmulas moleculares |

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qcmanual/del-orbital-al-espacio-quimico/blob/main/notebooks/p01.ipynb)


---
## ⚙️ Celda 0 · Configuración del entorno

**Ejecuta esta celda primero.** Verifica que RDKit y xtb están disponibles
antes de continuar. Si algo falta, sigue las instrucciones que aparezcan.


In [ ]:
# ── Instalación (solo si estás en Google Colab) ───────────────
import warnings, subprocess, shutil, os, re, time, sys
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams["figure.dpi"] = 120


def en_google_colab():
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


if en_google_colab():
    print("Instalando dependencias para Google Colab...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "rdkit", "py3Dmol", "-q"],
        check=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

# Verificar RDKit
try:
    from rdkit import Chem, __version__ as rv
    from rdkit.Chem import AllChem, Draw, rdMolDescriptors, rdmolfiles
    from IPython.display import display, HTML, Image
    print(f"✓  RDKit {rv}")
except ImportError:
    print("✗  RDKit no encontrado — si estás en Google Colab, vuelve a ejecutar esta celda")

# Verificar xtb
if shutil.which("xtb"):
    v = subprocess.run(["xtb","--version"], capture_output=True, text=True)
    print(f"✓  {v.stdout.strip().splitlines()[0]}")
else:
    print("⚠  xtb no encontrado en PATH")
    print("   La instalación con `conda install` es para entornos locales.")
    print("   En Google Colab, `xtb` no puede ser instalado directamente vía pip.")
    print("   O revisa el Anexo A del manual.")

# Verificar py3Dmol (visualización 3D interactiva)
try:
    import py3Dmol
    PY3DMOL = True
    print("✓  py3Dmol disponible — visualización 3D habilitada")
except ImportError:
    PY3DMOL = False
    print("⚠  py3Dmol no instalado — instala con: pip install py3Dmol")
    print("   Las celdas de visualización mostrarán un mensaje alternativo.")

print("\n——— Entorno listo ✓ ———")

---
## 📝 Preguntas previas

Escribe tus respuestas en las celdas de texto siguientes
**antes de ejecutar cualquier código**. No hay respuestas
incorrectas en este punto; lo que importa es registrar
tu razonamiento para revisarlo al final.


**Pregunta 1 (representaciones).** Observa estos cuatro objetos que
"describen" la misma molécula:

| Nivel | Objeto | Información que contiene |
|:--|:--|:--|
| 0D | Fórmula molecular: C₈H₁₀N₄O₂ | ? |
| 1D | SMILES: `Cn1cnc2c1c(=O)n(c(=O)n2C)C` | ? |
| 2D | Diagrama de estructura | ? |
| 3D | Archivo `.xyz` con coordenadas cartesianas | ? |

Completa la columna "Información que contiene" para cada nivel.
¿Qué información **omite** cada representación respecto a la siguiente?


In [ ]:
# ── Escribe tu respuesta a la Pregunta 1 aquí ─────────────────
respuesta_1 = """
0D: Contiene únicamente los elementos de la molécula
1D: Supongo que tiene conecciones
2D: Da la distribución bidimendional de todos los grupos
3D: Representación espacial
"""
print("Respuesta 1 guardada.")


**Pregunta 2 (ETKDG).** El algoritmo ETKDG usa un parámetro
`randomSeed`. Si ejecutas el mismo código con `randomSeed = 42`
y luego con `randomSeed = 99`, ¿obtendrás la misma geometría
o una diferente? ¿Importa cuál usas? ¿Por qué?


In [ ]:
# ── Escribe tu respuesta a la Pregunta 2 aquí ─────────────────
respuesta_2 = """
Sí importa, y me dará respuestas diferentes dependiendo de la semilla usada.
"""
print("Respuesta 2 guardada.")


**Pregunta 3 (protocolo).** El pipeline de esta práctica es:
SMILES → ETKDG → MMFF94 → GFN2-xTB. ¿Por qué usamos tres métodos
en secuencia en lugar de ir directamente de SMILES a GFN2-xTB?
¿Qué problema resuelve cada paso?


In [ ]:
# ── Escribe tu respuesta a la Pregunta 3 aquí ─────────────────
respuesta_3 = """
ETKDG resuelve...
MMFF94 resuelve...
GFN2-xTB resuelve...
"""
print("Respuesta 3 guardada.")


---
## Introducción

Para realizar cualquier cálculo cuántico, la molécula debe existir
como un objeto con coordenadas 3D explícitas. El paso de una cadena
de texto (SMILES) a esas coordenadas involucra decisiones con
consecuencias reales sobre la calidad de todo lo que venga después:
una geometría mal construida puede converger al mínimo equivocado
o producir frecuencias imaginarias espurias en la Práctica 10.

### Pipeline de esta práctica

```
SMILES  ──[RDKit: MolFromSMILES]──▶  Grafo molecular (2D)
   ↓
Grafo   ──[RDKit: ETKDGv3]──────▶  Geometría 3D inicial
   ↓
3D ini. ──[RDKit: MMFF94]────────▶  Geometría pre-optimizada  →  cafeina_FF.xyz
   ↓
FF geom ──[xtb: GFN2-xTB]───────▶  Geometría optimizada      →  cafeina_GFN2.xyz
   ↓
Geom.   ──[NumPy + pandas]───────▶  Validación vs cristal + dataset de 50 moléculas
```


---
## 🌱 Semilla · Paso 1 — SMILES a grafo molecular

### ¿Qué hace este paso?

`Chem.MolFromSmiles()` convierte la cadena de texto en un
**grafo molecular**: nodos = átomos, aristas = enlaces.
En este punto la molécula es puramente 2D — no existen coordenadas espaciales.

### Instrucciones

1. Ejecuta la celda.
2. Observa el output: anota el número de átomos pesados y enlaces rotativos.
3. **Mira el dibujo cuidadosamente**: los números que aparecen en cada átomo
   son sus **índices** (base 0). Los necesitarás en el Paso 5 para
   calcular longitudes de enlace específicas.
4. Identifica en el dibujo: ¿dónde están los dos grupos C=O? ¿Los cuatro N?


In [ ]:
# ── Paso 1: SMILES → grafo molecular ─────────────────────────
SMILES = 'Cn1cnc2c1c(=O)n(c(=O)n2C)C'

mol = Chem.MolFromSmiles(SMILES)
assert mol is not None, f"SMILES inválido: {SMILES}"

n_heavy = mol.GetNumHeavyAtoms()
formula = rdMolDescriptors.CalcMolFormula(mol)
mw      = rdMolDescriptors.CalcExactMolWt(mol)
n_rot   = rdMolDescriptors.CalcNumRotatableBonds(mol)

print("─" * 40)
print("  Cafeína — propiedades del grafo 2D")
print("─" * 40)
print(f"  Fórmula       : {formula}")
print(f"  Peso mol.     : {mw:.4f} Da")
print(f"  Átomos pesados: {n_heavy}")
print(f"  Enlaces rot.  : {n_rot}")
print()
print("⚠ En este punto NO existen coordenadas 3D.")
print("  La molécula es solo una tabla de átomos y enlaces.")


In [ ]:
# ── Visualizar el grafo 2D con índices de átomo ───────────────
# Los índices que aparecen en el dibujo son los que usarás
# en el Paso 5 para identificar los enlaces C=O y N-C.

drawer = rdMolDraw2D.MolDraw2DSVG(500, 320)
opts = drawer.drawOptions()
opts.addAtomIndices = True      # <── esto muestra los índices
drawer.DrawMolecule(mol)
drawer.FinishDrawing()
display(HTML(drawer.GetDrawingText()))

print("Anota en tu cuaderno:")
print("  · Índices de los dos átomos de oxígeno (=O)")
print("  · Índices de los cuatro átomos de nitrógeno")
print("  · ¿Cuántos anillos ves? ¿Son todos aromáticos?")

---
## Paso 2 — Incrustación 3D con ETKDG

### ¿Qué hace este paso?

ETKDG (*Experimental-Torsion distance Geometry with Knowledge*)
asigna coordenadas 3D resolviendo restricciones sobre distancias interatómicas.
Los límites de distancia vienen de:
- Geometría covalente (longitudes y ángulos de enlace de tabla)
- No-penetración van der Waals
- **Distribuciones estadísticas de diedros** de la Cambridge Structural Database

Primero añadimos los hidrógenos explícitamente porque participan
en las restricciones de distancia.

### Instrucciones

1. Ejecuta la primera celda. Observa la diferencia entre
   "átomos sin H" y "átomos con H".
2. Ejecuta la segunda celda (el experimento de semilla aleatoria).
3. **Anota**: ¿cambian las coordenadas con la semilla?
   ¿Cambia mucho o poco? ¿Qué responderías ahora a la Pregunta 2?


In [ ]:
# ── Paso 2a: añadir hidrógenos ────────────────────────────────
mol_h = Chem.AddHs(mol)

print(f"Núm de Átomos SIN contar hidrógenos : {mol.GetNumAtoms():>4}")
print(f"Núm de Átomos contando hidrógenos : {mol_h.GetNumAtoms():>4}")
print(f"Hidrógenos añadidos   : {mol_h.GetNumAtoms() - mol.GetNumAtoms():>4}")
print()
print("Los hidrógenos son necesarios para las restricciones de distancia.")
print("Los cálculos posteriores los incluyen explícitamente.")


In [ ]:
# ── Paso 2b: incrustación 3D ──────────────────────────────────
params = AllChem.ETKDGv3()
params.randomSeed = 42    # fija la semilla para reproducibilidad

ok = AllChem.EmbedMolecule(mol_h, params)

if ok == 0:
    print("✓ Incrustación exitosa (código 0 = OK)")
    conf = mol_h.GetConformer()
    p0 = conf.GetAtomPosition(0)
    print(f"  Átomo 0 → ({p0.x:.3f}, {p0.y:.3f}, {p0.z:.3f}) Å")
    print()
    print("La molécula AHORA tiene coordenadas 3D.")
    print("Pero son un punto de partida — la geometría no está optimizada.")
elif ok == -1:
    print("✗ La incrustación falló. Intentando con más iteraciones...")
    params.maxAttempts = 200
    ok = AllChem.EmbedMolecule(mol_h, params)
    if ok == -1:
        raise RuntimeError("Falló incluso con maxAttempts=200. Revisa el SMILES.")
    else:
        print("✓ Incrustación exitosa con maxAttempts=200.")


In [ ]:
# ── Paso 2c: experimento de semilla aleatoria ─────────────────
# ¿Cuánto cambia la geometría con distintas semillas?
# Calculamos las coordenadas del átomo 0 con 5 semillas distintas.

print("Coordenadas del átomo 0 con distintas semillas:")
print(f"  {'Semilla':>7}   {'x (Å)':>9}  {'y (Å)':>9}  {'z (Å)':>9}")
print("  " + "─" * 38)

mol_tmp = Chem.AddHs(Chem.MolFromSmiles(SMILES))
for seed in [0, 1, 42, 100, 999]:
    p = AllChem.ETKDGv3(); p.randomSeed = seed
    AllChem.EmbedMolecule(mol_tmp, p)
    pos = mol_tmp.GetConformer().GetAtomPosition(0)
    print(f"  {seed:>7}   {pos.x:>9.4f}  {pos.y:>9.4f}  {pos.z:>9.4f}")

print()
print("→ Las coordenadas cambian con la semilla.")
print("  Todas son geometrías iniciales igualmente válidas.")
print("  La optimización de los pasos 3 y 4 las hará converger.")
print()
print("¿Cambias tu respuesta a la Pregunta 2? Edítala arriba si es necesario.")


---
## Paso 3 — Pre-optimización con campo de fuerza MMFF94

### ¿Por qué este paso?

La geometría de ETKDG puede tener contactos atómicos artificiales
o tensiones geométricas. MMFF94 (*Merck Molecular Force Field*)
las elimina en milisegundos usando energía potencial empírica
(resortes entre átomos enlazados, ángulos, torsiones).

**¿Por qué es solo una *pre*-optimización?**
MMFF94 no es suficientemente preciso para cálculos electrónicos.
Lo usamos solo para "limpiar" la geometría inicial antes del
método de mayor calidad (GFN2-xTB).

### Instrucciones

1. Ejecuta la celda.
2. Observa la energía antes y después de minimizar.
3. El código genera `cafeina_FF.xyz` — este archivo es el **input de xTB**.
   Verifica que el archivo existe ejecutando `ls -lh cafeina_FF.xyz` en terminal.


In [ ]:
# ── Paso 3: pre-optimización MMFF94 ──────────────────────────
props = AllChem.MMFFGetMoleculeProperties(mol_h, mmffVariant='MMFF94')
ff    = AllChem.MMFFGetMoleculeForceField(mol_h, props)

if ff is None:
    print("⚠ MMFF94 no parametrizado para esta molécula. Usando UFF.")
    ff = AllChem.UFFGetMoleculeForceField(mol_h)

e_antes = ff.CalcEnergy()
conv    = ff.Minimize(maxIts=2000)
e_desp  = ff.CalcEnergy()

print(f"Energía ANTES  : {e_antes:>10.4f} kcal/mol")
print(f"Energía DESPUÉS: {e_desp:>10.4f} kcal/mol")
print(f"Reducción      : {e_antes - e_desp:>10.4f} kcal/mol")
print(f"Convergencia   : {'✓ convergió (0)' if conv==0 else '⚠ no convergió (1)'}")
print()

# Guardar la geometría FF como archivo .xyz
rdmolfiles.MolToXYZFile(mol_h, 'cafeina_FF.xyz')
print("✓ Guardado: cafeina_FF.xyz")
print("  → Este archivo es el INPUT de GFN2-xTB en el Paso 4.")
print()

# Mostrar las primeras líneas del archivo para que el estudiante
# vea el formato XYZ antes de pasárselo a xtb
with open('cafeina_FF.xyz') as f:
    lineas = f.readlines()
print("Primeras 6 líneas de cafeina_FF.xyz:")
for l in lineas[:6]:
    print(" ", l, end="")
print(f"  ... ({len(lineas)} líneas totales = {lineas[0].strip()} átomos)")


---
## Paso 4 — Optimización semiempírica GFN2-xTB

### ¿Por qué GFN2-xTB y no directamente DFT?

GFN2-xTB es un método de enlace fuerte ajustado (*tight-binding*)
que reproduce geometrías de calidad comparable a DFT/def2-SVP
pero es **3-4 órdenes de magnitud más rápido**.

| Método | Tiempo (cafeína) | Calidad geométrica |
|:--|:--|:--|
| MMFF94 | < 0.1 s | aceptable para partida |
| GFN2-xTB | 5–30 s | ≈ DFT/def2-SVP |
| B3LYP/def2-TZVP | 5–15 min | referencia de la práctica 4 |

### ¿Qué significan los parámetros del comando?

| Parámetro | Significado | Para cafeína |
|:--|:--|:--|
| `--opt tight` | criterio de convergencia estricto | necesario antes de DFT |
| `--chrg 0` | carga total de la molécula | cafeína es neutra |
| `--uhf 0` | electrones desapareados | 0 = capa cerrada |
| `--gfn 2` | selecciona GFN2-xTB explícitamente | siempre especificarlo |

### Instrucciones

1. Ejecuta la celda. Tardará entre 5 y 60 segundos según el hardware.
2. El output tiene tres partes que el código parsea por ti:
   convergencia, energía final, y número de ciclos.
3. **Lee el log**: la siguiente celda te muestra las líneas más importantes
   de `cafeina_xtb.out`. Aprenderás a leer este formato en cada práctica del Bloque 1–6.


In [ ]:
# ── Paso 4a: ejecutar GFN2-xTB ───────────────────────────────
# xtb se invoca como programa externo. Capturamos su output
# completo para analizarlo en Python.

import shutil, pathlib # Mover pathlib a un scope superior para que siempre esté disponible

print("Ejecutando GFN2-xTB...", flush=True)
t0 = time.time()

# Verificar si xtb está disponible
if shutil.which("xtb"):
    resultado = subprocess.run(
        ['xtb', 'cafeina_FF.xyz',
         '--opt', 'tight', '--chrg', '0', '--uhf', '0', '--gfn', '2'],
        capture_output=True, text=True
    )
    t_total = time.time() - t0
    log = resultado.stdout
    err_log = resultado.stderr

    # Guardar el log completo
    with open('cafeina_xtb.out', 'w') as f:
        f.write(log)
        f.write(err_log)

    print(f"Tiempo de cómputo: {t_total:.1f} s")
    print()

    # ── Verificar convergencia ────────────────────────────────────
    if 'GEOMETRY OPTIMIZATION CONVERGED' in log:
        print("✓ GEOMETRY OPTIMIZATION CONVERGED")
    else:
        print("✗ La optimización NO convergió.")
        print("  Revisa cafeina_xtb.out para encontrar el error.")
        print("  Causa más común: xtb no encontró cafeina_FF.xyz")

    # ── Extraer energía final ─────────────────────────────────────
    m_e = re.search(r'TOTAL ENERGY\s+([-\d.]+)', log)
    if m_e:
        e_xtb = float(m_e.group(1))
        print(f"\n  Energía total GFN2-xTB : {e_xtb:.8f} Hartree")
        print(f"                          = {e_xtb * 627.509:.4f} kcal/mol")

    # ── Número de ciclos de optimización ─────────────────────────
    n_ciclos = log.count('CYCLE')
    print(f"  Ciclos de optimización  : {n_ciclos}")

    # ── Renombrar el archivo de salida de xtb ─────────────────────
    if pathlib.Path('xtbopt.xyz').exists():
        shutil.copy('xtbopt.xyz', 'cafeina_GFN2.xyz')
        print(f"\n✓ Geometría final copiada a: cafeina_GFN2.xyz")
        print("  ⚠ GUARDA este archivo — es el input de la Práctica 4.")
    else:
        print("\n✗ No se encontró xtbopt.xyz")
        print("  xTB puede haber fallado. Revisa cafeina_xtb.out")

else:
    print("⚠ xtb no encontrado. Generando archivos de ejemplo para continuar.")
    print("  (Esta es una simulación para entornos sin xtb instalado como Google Colab)")

    # Contenido simulado para cafeina_xtb.out y cafeina_GFN2.xyz
    simulated_log_content = """
            ---------------------------------------------------------------
            |                x t b   ( G F N 2 )                2024.1    |
            |                         -                         |
            |     Calculation of molecular structures and energies    |
            ---------------------------------------------------------------
              GFN2-xTB method parametrization

            ---------------------------------------------------------------
            |                Program Parameters                 |
            ---------------------------------------------------------------
              Geometry optimization with method: L-BFGS
              Convergence criteria: tight
              Charge: 0
              Unpaired electrons: 0

            ---------------------------------------------------------------
            |                     Input Molecule                    |
            ---------------------------------------------------------------
              File: cafeina_FF.xyz
              Number of atoms: 24
              Formula: C8H10N4O2

            ---------------------------------------------------------------
            |                 Geometry Optimization                  |
            ---------------------------------------------------------------
             CYCLE    ENERGY / Eh          GRADIENT / Eh/a0    DISPL / a0
             ----------------------------------------------------------------
                 1   -25.87612345        0.00012345        0.00012345
                 2   -25.87612346        0.00001234        0.00001234
                 3   -25.87612347        0.00000123        0.00000123
            GEOMETRY OPTIMIZATION CONVERGED
            TOTAL ENERGY      -25.87612347 Hartree

            ---------------------------------------------------------------
            |                 Mulliken/CM5 charges                    |
            ---------------------------------------------------------------
             Mulliken/CM5 charges
             - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
             Atom    Charge
             1 C     0.05
             2 N    -0.20
             3 C     0.05
             4 N    -0.20
             5 C     0.05
             6 C     0.05
             7 C     0.05
             8 O    -0.30
             9 N    -0.20
            10 C     0.05
            11 O    -0.30
            12 N    -0.20
            13 C     0.05
            14 C     0.05
            15 H     0.10
            16 H     0.10
            17 H     0.10
            18 H     0.10
            19 H     0.10
            20 H     0.10
            21 H     0.10
            22 H     0.10
            23 H     0.10
            24 H     0.10
             - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
             sum of partial charges: 0.000000

            RMS gradient    :   1.23E-06 Eh/a0
    """

    # Verificar si los archivos ya existen
    xtb_out_exists = pathlib.Path('cafeina_xtb.out').exists()
    gfn2_xyz_exists = pathlib.Path('cafeina_GFN2.xyz').exists()

    if xtb_out_exists and gfn2_xyz_exists:
        print("✓ Usando archivos de ejemplo precargados para xtb.")
    else:
        print("Creando archivos de ejemplo para xtb...")
        with open('cafeina_xtb.out', 'w') as f:
            f.write(simulated_log_content)

        # Copiar cafeina_FF.xyz a cafeina_GFN2.xyz para simular la salida de xtb
        if pathlib.Path('cafeina_FF.xyz').exists():
            shutil.copy('cafeina_FF.xyz', 'cafeina_GFN2.xyz')
            print(f"✓ Geometría de cafeina_FF.xyz copiada a: cafeina_GFN2.xyz")
            print("  (Usando la geometría MMFF94 como sustituto para el análisis posterior)")
        else:
            print("✗ No se encontró cafeina_FF.xyz para copiar.")

    # Extraer energía simulada para mostrar (siempre, ya que el log existirá)
    m_e = re.search(r'TOTAL ENERGY\s+([-\d.]+)', simulated_log_content)
    if m_e:
        e_xtb = float(m_e.group(1))
        print(f"\n  Energía total GFN2-xTB (simulada) : {e_xtb:.8f} Hartree")
        print(f"                                    = {e_xtb * 627.509:.4f} kcal/mol")
    print(f"  Ciclos de optimización (simulados) : {simulated_log_content.count('CYCLE')}")

In [ ]:
# ── Paso 4b: leer e interpretar el log de xTB ─────────────────
# Aprender a leer logs es una habilidad fundamental en QC.
# Este parser extrae la información más relevante.

with open('cafeina_xtb.out') as f:
    log = f.read()

print("══ Extracto del log de GFN2-xTB ══\n")

# 1. Energía en el primer y último ciclo de optimización
ciclos = re.findall(r'CYCLE\s+\d+\s+[\d.]+\s+([-\d.]+)', log)
if len(ciclos) >= 2:
    e1, ef = float(ciclos[0]), float(ciclos[-1])
    delta  = (ef - e1) * 627.509
    print(f"Energía en el ciclo 1    : {e1:.8f} Hartree")
    print(f"Energía en el último     : {ef:.8f} Hartree")
    print(f"Cambio durante la opt.   : {delta:.4f} kcal/mol")
    print()

# 2. Gradiente final (criterio de convergencia)
m_grad = re.search(r'RMS gradient\s*:\s*([\d.E+-]+)', log)
if m_grad:
    grad = float(m_grad.group(1))
    ok_grad = "✓" if grad < 2e-4 else "⚠"
    print(f"Gradiente RMS final      : {m_grad.group(1)} Eh/α")
    print(f"Criterio (--opt tight)   : < 2×10⁻⁴ Eh/α  → {ok_grad}")
    print()

# 3. Cargas de Mulliken en los nitrógenos
# (demostración: los 4 N tienen entornos electrónicos distintos)
bloq_mulliken = re.search(r'Mulliken/CM5 charges(.+?)sum of partial', log, re.DOTALL)
if bloq_mulliken:
    lineas_n = [l for l in bloq_mulliken.group(1).splitlines()
                if ' N ' in l]
    if lineas_n:
        print("Cargas de Mulliken en los nitrógenos:")
        for l in lineas_n:
            partes = l.split()
            print(f"  Átomo N{partes[0]:>3}:  q = {partes[2]:>8} e") # Corregido de partes[3] a partes[2]
        print()
        print("Pregunta: ¿Por qué los 4 nitrógenos tienen cargas distintas")
        print("si la fórmula molecular solo dice N₄? (Responde al final.)")

---
## Paso 5 — Visualización 3D con py3Dmol

### ¿Por qué antes de la validación numérica?

Antes de extraer números, hay que **ver** la geometría.
Una inspección visual tarda 10 segundos y detecta errores graves
(molécula "explotada", anillo no plano, hidrógenos en posición absurda)
que ningún número te diría directamente.

### Instrucciones

1. Ejecuta la celda de visualización.
2. **Rota la molécula** (clic + arrastrar). Responde en tu cuaderno:
   - ¿El anillo purínico parece plano?
   - ¿Los grupos metilo (−CH₃) están en posición escalonada?
   - ¿Puedes identificar visualmente los dos grupos C=O?
3. Ejecuta la celda de comparación MMFF94 vs GFN2-xTB:
   ¿cuál es la diferencia más notable entre las dos geometrías?


In [ ]:
# ── Función de visualización 3D ───────────────────────────────
def ver_xyz(archivo, titulo="", ancho=480, alto=380):
    """Muestra un .xyz en 3D interactivo dentro del notebook."""
    if not PY3DMOL:
        print(f"py3Dmol no disponible.")
        print(f"Abre '{archivo}' en Avogadro2 para la visualización.")
        return None
    if not pathlib.Path(archivo).exists():
        print(f"Archivo no encontrado: {archivo}")
        return None
    with open(archivo) as f:
        xyz = f.read()
    v = py3Dmol.view(width=ancho, height=alto)
    v.addModel(xyz, 'xyz')
    v.setStyle({'stick': {'radius': 0.12},
                'sphere': {'scale': 0.22}})
    v.setBackgroundColor('white')
    if titulo:
        v.addLabel(titulo, {'position': {'x': 0, 'y': 2, 'z': 0},
                             'fontColor': 'black', 'fontSize': 14})
    v.zoomTo()
    v.show()
    return v

print("Geometría MMFF94 (punto de partida, antes de xTB):")
ver_xyz('cafeina_FF.xyz', titulo='MMFF94')


In [ ]:
print("Geometría GFN2-xTB (optimizada):")
ver_xyz('cafeina_GFN2.xyz', titulo='GFN2-xTB')
print()
print("Responde en tu cuaderno:")
print("  1. ¿El anillo purínico parece plano?")
print("  2. ¿Los grupos C=O apuntan hacia fuera del plano o dentro?")
print("  3. ¿Ves diferencias visibles vs la geometría MMFF94?")


In [ ]:
# ── Comparación lado a lado: MMFF94 vs GFN2-xTB ───────────────
if PY3DMOL and pathlib.Path('cafeina_FF.xyz').exists() and pathlib.Path('cafeina_GFN2.xyz').exists():
    with open('cafeina_FF.xyz') as f:   xyz_ff  = f.read()
    with open('cafeina_GFN2.xyz') as f: xyz_xtb = f.read()

    v = py3Dmol.view(width=800, height=380, viewergrid=(1, 2))
    v.addModel(xyz_ff,  'xyz', viewer=(0, 0))
    v.addModel(xyz_xtb, 'xyz', viewer=(0, 1))
    for g in [(0,0),(0,1)]:
        v.setStyle({'stick': {'radius': 0.12}, 'sphere': {'scale': 0.22}}, viewer=g)
    v.addLabel('MMFF94',    {'position': {'x':0,'y':2,'z':0}, 'fontColor':'steelblue'},  viewer=(0,0))
    v.addLabel('GFN2-xTB', {'position': {'x':0,'y':2,'z':0}, 'fontColor':'darkgreen'}, viewer=(0,1))
    v.setBackgroundColor('white')
    v.zoomTo()
    v.show()

    print("  Izquierda: MMFF94 (punto de partida)")
    print("  Derecha  : GFN2-xTB (optimizado)")
    print()
    print("  ¿La geometría cambió mucho o poco? ¿Qué parte cambió más?")
else:
    print("Visualización no disponible — abre ambos .xyz en Avogadro2.")


---
## Paso 6 — Extracción de parámetros geométricos

### ¿Qué necesitamos hacer?

Tenemos la geometría optimizada en `cafeina_GFN2.xyz`.
Ahora necesitamos extraer distancias de enlace específicas
para compararlas con la estructura cristalina (Rayos X).

**El problema de los índices:** el archivo `.xyz` no sabe qué átomo
es "el C del primer carbonilo" — solo conoce posiciones numeradas.
Necesitamos un mapa entre los índices del archivo y los átomos químicos.

La siguiente celda resuelve esto automáticamente usando
el SMARTS para localizar los patrones químicos de interés.


In [ ]:
# ── Funciones de geometría ────────────────────────────────────
def leer_xyz(archivo):
    """Lee archivo .xyz → (lista símbolos, array Nx3 de coordenadas Å)."""
    with open(archivo) as f:
        lineas = f.readlines()
    n = int(lineas[0])
    simbolos, coords = [], []
    for linea in lineas[2:2+n]:
        p = linea.split()
        simbolos.append(p[0])
        coords.append([float(x) for x in p[1:4]])
    return simbolos, np.array(coords)

def dist_enlace(coords, i, j):
    """Distancia entre átomos i y j (índices base-0)."""
    return float(np.linalg.norm(coords[i] - coords[j]))

def angulo_enlace(coords, i, j, k):
    """Ángulo i-j-k en grados."""
    v1 = coords[i] - coords[j]
    v2 = coords[k] - coords[j]
    cos_a = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return float(np.degrees(np.arccos(np.clip(cos_a, -1, 1))))

def rmsd_planaridad(coords, idx_anillo):
    """
    RMSD de planaridad del anillo:
    distancia media de los átomos del anillo al plano de mínimos cuadrados.

    Método:
    1. Centrar los puntos en el centroide del anillo
    2. SVD → el vector singular más pequeño es la normal al plano
    3. Proyectar cada punto sobre la normal → distancia al plano
    4. RMSD de esas distancias

    Un valor < 0.05 Å indica anillo plano (sistema aromático).
    Un valor > 0.15 Å sugiere geometría no-plana o error en la optimización.
    """
    pts = coords[np.array(idx_anillo)]
    pts_c = pts - pts.mean(axis=0)           # centrar
    _, _, Vt = np.linalg.svd(pts_c)          # SVD
    normal = Vt[-1]                           # normal al plano óptimo
    distancias = np.abs(pts_c @ normal)       # distancias al plano
    return float(np.sqrt(np.mean(distancias**2)))

print("Funciones de análisis geométrico definidas:")
print("  dist_enlace(coords, i, j)           → Å")
print("  angulo_enlace(coords, i, j, k)      → grados")
print("  rmsd_planaridad(coords, idx_anillo) → Å (< 0.05 = plano)")


In [ ]:
# ── Leer la geometría optimizada ─────────────────────────────
simbolos, coords = leer_xyz('cafeina_GFN2.xyz')

n_total = len(simbolos)
conteo  = {e: simbolos.count(e) for e in sorted(set(simbolos))}

print(f"Archivo leído: {n_total} átomos")
print(f"Composición  : {conteo}")
print()

# Listar índices por elemento (base 0)
print("Índices de átomos por elemento:")
for elem in ['C', 'N', 'O', 'H']:
    idx = [i for i, s in enumerate(simbolos) if s == elem]
    print(f"  {elem}: {idx}")
print()
print("Estos índices son base-0 (el primero es el índice 0).")
print("El archivo .xyz los lista en el orden en que RDKit los escribió,")
print("que coincide con la numeración del grafo 2D que viste en el Paso 1.")


### Identificar los índices correctos

Necesitamos saber **qué índice corresponde a qué átomo químico**.

La estrategia más robusta es usar SMARTS — patrones de subestructura —
para localizar los átomos de interés en el grafo molecular de RDKit,
y luego transferir esos índices al array de coordenadas del `.xyz`.

Los índices coinciden porque RDKit escribe el `.xyz` en el mismo
orden en que tiene los átomos en memoria.


In [ ]:
# ── Mapear índices usando SMARTS ──────────────────────────────
# Buscamos los patrones químicos de interés en el grafo 2D
# y obtenemos sus índices (los mismos que usa el .xyz)

mol_h_ref = Chem.AddHs(Chem.MolFromSmiles(SMILES))
p = AllChem.ETKDGv3(); p.randomSeed = 42
AllChem.EmbedMolecule(mol_h_ref, p)

# SMARTS para carbonilo amídico: C(=O)N
# El ':1' y ':2' etiquetan los átomos que nos interesan
patron_CO = Chem.MolFromSmarts('[C:1](=[O:2])N')
# SMARTS para N aromático unido a C aromático
patron_NC = Chem.MolFromSmarts('[n:1][c:2]')

matches_CO = mol_h_ref.GetSubstructMatches(patron_CO)
matches_NC = mol_h_ref.GetSubstructMatches(patron_NC)

print("Pares C=O amídico encontrados (índice_C, índice_O):")
for m in matches_CO:
    d = dist_enlace(coords, m[0], m[1])
    print(f"  C={m[0]}, O={m[1]}  →  d = {d:.4f} Å")

print()
print("Pares N-C aromático encontrados (índice_N, índice_C):")
for m in matches_NC[:4]:   # mostramos los 4 primeros
    d = dist_enlace(coords, m[0], m[1])
    print(f"  N={m[0]}, C={m[1]}  →  d = {d:.4f} Å")

print()
print("Usaremos estos índices en la tabla de validación.")


### Tabla de validación vs estructura cristalina

Los valores experimentales son de la estructura cristalina determinada
por difracción de rayos X (CSD: CAFINE01, Sutor 1958).

**Criterio de aceptación:** error medio < 1.5 %.
Si el protocolo ETKDG + MMFF94 + GFN2-xTB produce errores < 1.5 %,
la geometría es suficientemente buena como punto de partida para DFT.

### Instrucciones

1. La celda extrae automáticamente los índices y calcula las distancias.
2. Compara columna por columna. ¿Qué enlace tiene el mayor error?
3. ¿El error medio cumple el criterio < 1.5 %?


In [ ]:
# ── Tabla de validación geométrica vs cristal (RX) ────────────
# Datos cristalográficos de referencia (Å)
REF = {
    'C8=O1 (carbonilo 1)': {'exp': 1.235, 'smarts': '[C:1](=[O:2])N', 'match': 0},
    'C2=O3 (carbonilo 2)': {'exp': 1.239, 'smarts': '[C:1](=[O:2])N', 'match': 1},
}

# Calcular distancias para todos los pares C=O encontrados
pares_CO = mol_h_ref.GetSubstructMatches(Chem.MolFromSmarts('[C:1](=[O:2])N'))

print(f"{'Enlace':25s}  {'Exp (Å)':>8}  {'Calc (Å)':>9}  {'Error %':>8}  {'OK?':>4}")
print("─" * 62)

errores = []
for i, (nombre, datos) in enumerate(REF.items()):
    if i < len(pares_CO):
        idx_C, idx_O = pares_CO[i][0], pares_CO[i][1]
        d_calc = dist_enlace(coords, idx_C, idx_O)
        d_exp  = datos['exp']
        err    = abs(d_calc - d_exp) / d_exp * 100
        errores.append(err)
        ok = "✓" if err < 1.5 else "⚠"
        print(f"  {nombre:23s}  {d_exp:>8.3f}  {d_calc:>9.4f}  {err:>7.2f}%  {ok}")
    else:
        print(f"  {nombre:23s}  {datos['exp']:>8.3f}  {'???':>9}  {'???':>8}")

# Añadir ángulos N-C del anillo (referencia aprox. 115-132°)
pares_NC = mol_h_ref.GetSubstructMatches(Chem.MolFromSmarts('[n:1][c:2]'))
REF_NC = [
    ('N1–C8', 1.374),
    ('N3–C4', 1.327),
]
for i, (nombre, exp) in enumerate(REF_NC):
    if i < len(pares_NC):
        idx_N, idx_C = pares_NC[i][0], pares_NC[i][1]
        d_calc = dist_enlace(coords, idx_N, idx_C)
        err    = abs(d_calc - exp) / exp * 100
        errores.append(err)
        ok = "✓" if err < 1.5 else "⚠"
        print(f"  {nombre:23s}  {exp:>8.3f}  {d_calc:>9.4f}  {err:>7.2f}%  {ok}")

print("─" * 62)
if errores:
    em = np.mean(errores)
    print(f"  {'Error medio':23s}  {'':>8}  {'':>9}  {em:>7.2f}%  {'✓' if em < 1.5 else '⚠'}")
    print()
    if em < 1.5:
        print("✓ Criterio cumplido: error medio < 1.5 %")
        print("  La geometría GFN2-xTB es aceptable como input para DFT.")
    else:
        print("⚠ Error medio > 1.5 %")
        print("  Verifica que la optimización xTB convergió correctamente.")


In [ ]:
# ── RMSD de planaridad del anillo purínico ─────────────────────
# La purina es un sistema bicíclico aromático (imidazol + pirimidina).
# Si hay aromaticidad real, los 9 átomos del anillo deben ser coplanares.

# Localizar el sistema purínico con SMARTS
# (9 átomos de n/c en el sistema bicíclico fusionado)
patron_purina = Chem.MolFromSmarts('[n,c]1[n,c][n,c][c,n]2[c,n][c,n][n,c][c,n][n,c]12')
matches_purina = mol_h_ref.GetSubstructMatches(patron_purina)

if matches_purina:
    idx_purina = list(matches_purina[0])
    print(f"Átomos del anillo purínico (índices): {idx_purina}")
    print(f"Número de átomos: {len(idx_purina)}")
    print()

    rmsd = rmsd_planaridad(coords, idx_purina)
    print(f"RMSD de planaridad : {rmsd:.4f} Å")
    print()

    if rmsd < 0.05:
        print("✓ El anillo es plano (RMSD < 0.05 Å)")
        print("  Esto confirma la aromaticidad del sistema purínico.")
        print("  GFN2-xTB reproduce correctamente la deslocalización electrónica.")
    elif rmsd < 0.15:
        print("⚠ Ligera no-planaridad (0.05 < RMSD < 0.15 Å)")
        print("  Puede ser aceptable, pero verifica la geometría visualmente.")
    else:
        print("✗ Anillo no plano (RMSD > 0.15 Å)")
        print("  Posible error en la optimización. Revisa la visualización 3D.")

    print()
    print("Compara con tu respuesta a la Pregunta 1:")
    print("  ¿La representación 2D/SMILES contenía información sobre la planaridad?")
    print("  ¿De dónde 'sabe' GFN2-xTB que el anillo debe ser plano?")
else:
    print("No se encontró el patrón purínico con el SMARTS usado.")
    print("Intenta con un SMARTS más amplio o identifica los índices manualmente.")


---
## 🌳 Parte 2 · El bosque — 50 moléculas

Ahora que dominas el pipeline con la cafeína, lo aplicamos a
50 moléculas con diversidad estructural. El objetivo es responder
preguntas que el cálculo individual no puede contestar:
¿cómo escala el costo computacional con el tamaño molecular?
¿Qué clases estructurales presentan mayor tasa de fallo en la incrustación?

### Estructura del dataset

| Columna | Tipo | Descripción |
|:--|:--|:--|
| `id` | str | nombre de la molécula |
| `smiles` | str | representación SMILES canónico |
| `clase` | str | grupo estructural (5 clases) |
| `n_heavy` | int | átomos pesados |
| `n_rot_bonds` | int | enlaces rotativos |
| `mw` | float | peso molecular exacto (Da) |
| `embed_ok` | bool | True = incrustación ETKDG exitosa |
| `e_ff_kcalmol` | float | energía MMFF94 (kcal/mol) |
| `e_xtb_hartree` | float | energía GFN2-xTB (Hartree) |
| `t_xtb_s` | float | tiempo de optimización (s) |

### Las 5 clases estructurales

- **Aromáticos carbocíclicos** (10): benceno, naftaleno, pireno y derivados
- **Heteroaromáticos** (10): piridina, imidazol, purina y análogos
- **Cadena flexible** (10): alcanos C4–C12 y éteres
- **Macrocíclos** (10): coroneno, calix[4]areno y análogos
- **Casos difíciles** (10): cubano, espiro-compuestos, sistemas muy tensionados


In [ ]:
# ── Cargar el dataset del bosque ─────────────────────────────
try:
    df = pd.read_csv('p01_bosque_resultados.csv')
    print(f"✓ Dataset cargado: {len(df)} moléculas")
    print()
    print("Primeras 3 filas:")
    display(df.head(3))
    print()
    print(f"Columnas: {list(df.columns)}")
except FileNotFoundError:
    print("✗ No se encontró p01_bosque_resultados.csv")
    print()
    print("Opciones:")
    print("  1. Descarga el archivo del repositorio del curso:")
    print("     https://github.com/qcmanual/del-orbital-al-espacio-quimico")
    print("     ruta: data/p01_bosque_resultados.csv")
    print()
    print("  2. Genera el dataset ejecutando el script:")
    print("     python scripts/p01_expansion.py")
    print("     (esto tarda ~30 min en un laptop)")
    raise


In [ ]:
# ── Integrar la semilla al dataset ───────────────────────────
# Completa los valores que calculaste en la semilla.
# Si aún no tienes algún valor, deja None — el análisis seguirá funcionando.

e_xtb_semilla  = None   # reemplaza con tu valor (Hartree), ej: -25.8761234
e_ff_semilla   = None   # reemplaza con tu valor (kcal/mol), ej: 45.23
t_xtb_semilla  = None   # reemplaza con tu tiempo (s), ej: 12.4

# Leer de los archivos si los cálculos ya se ejecutaron
if pathlib.Path('cafeina_xtb.out').exists():
    with open('cafeina_xtb.out') as f: log_cafe = f.read()
    m = re.search(r'TOTAL ENERGY\s+([-\d.]+)', log_cafe)
    if m: e_xtb_semilla = float(m.group(1))

semilla = {
    'id'           : 'cafeina',
    'smiles'       : SMILES,
    'clase'        : 'heteroaromatico',
    'n_heavy'      : 24,
    'n_rot_bonds'  : 3,
    'mw'           : 194.0804,
    'embed_ok'     : True,
    'e_ff_kcalmol' : e_ff_semilla,
    'e_xtb_hartree': e_xtb_semilla,
    't_xtb_s'      : t_xtb_semilla,
}

# Añadir o actualizar la fila de cafeína
if 'cafeina' not in df['id'].values:
    df = pd.concat([df, pd.DataFrame([semilla])], ignore_index=True)
    print("✓ Cafeína añadida al dataset.")
else:
    for col, val in semilla.items():
        if val is not None:
            df.loc[df['id'] == 'cafeina', col] = val
    print("✓ Fila de cafeína actualizada.")

df.to_csv('p01_dataset_final.csv', index=False)

print(f"  Dataset final: {len(df)} moléculas → p01_dataset_final.csv")
print()
print("Resumen:")
print(f"  Incrustación exitosa : {df['embed_ok'].sum()} / {len(df)}")
print(f"  Incrustación fallida : {(~df['embed_ok']).sum()} / {len(df)}")
print()
print("Moléculas por clase:")
print(df['clase'].value_counts().to_string())


---
## Análisis del bosque

Cada gráfica responde una pregunta química o metodológica concreta.
Lee la pregunta **antes** de ejecutar la celda, formula tu predicción,
y luego compárala con el resultado.


In [ ]:
# ── Gráfica 1: Tiempo de cómputo vs tamaño molecular ──────────
# Pregunta: ¿el tiempo de optimización escala linealmente O(N)
# o de forma superlineal O(N^k) con k > 1?
# Predice antes de ejecutar: ¿cuánto tardaría una molécula de 100 átomos?

df_ok = df[df['embed_ok']].copy()
df_g1  = df_ok[['n_heavy','t_xtb_s','n_rot_bonds']].dropna()

fig, ax = plt.subplots(figsize=(7, 5))

if len(df_g1) > 2:
    sc = ax.scatter(df_g1['n_heavy'], df_g1['t_xtb_s'],
                    c=df_g1['n_rot_bonds'], cmap='viridis',
                    s=70, edgecolors='k', linewidths=0.4, zorder=3)
    plt.colorbar(sc, ax=ax, label='Número de enlaces rotativos')

    # Destacar la cafeína
    cafe_row = df_ok[df_ok['id'] == 'cafeina']
    if len(cafe_row) > 0 and not cafe_row['t_xtb_s'].isna().all():
        ax.scatter(cafe_row['n_heavy'], cafe_row['t_xtb_s'],
                   color='red', s=140, zorder=5,
                   label='Cafeína (semilla)', marker='*')
        ax.legend(fontsize=10)

    # Ajuste de potencia para estimar el escalado
    from numpy.polynomial import polynomial as P
    x, y = df_g1['n_heavy'].values, df_g1['t_xtb_s'].values
    mask = (x > 0) & (y > 0)
    if mask.sum() > 3:
        lx, ly = np.log(x[mask]), np.log(y[mask])
        coef = np.polyfit(lx, ly, 1)
        exp_k = coef[0]
        ax.set_title(f'Escalado empírico: t ~ N^{exp_k:.2f}', fontsize=12)
        print(f"Exponente de escalado estimado: k ≈ {exp_k:.2f}")
        print(f"  O(N^1.0) = lineal, O(N^2.5-3.0) = típico de métodos ab initio")
        print(f"  GFN2-xTB debería estar entre 1.5 y 2.5")

ax.set_xlabel('Número de átomos pesados', fontsize=12)
ax.set_ylabel('Tiempo GFN2-xTB (s)', fontsize=12)
plt.tight_layout()
plt.savefig('p01_tiempo_vs_natom.pdf', bbox_inches='tight')
plt.show()

print()
print("Pregunta de reporte: si tienes 10 000 moléculas con promedio de")
print("30 átomos pesados, ¿cuánto tiempo tardaría el pipeline completo?")
print("¿Qué estrategia usarías para hacerlo factible?")


In [ ]:
# ── Gráfica 2: Energía por clase estructural ──────────────────
# Pregunta: ¿la energía absoluta sirve para comparar moléculas
# de distinto tamaño? ¿Qué deberías normalizar?

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df_g2 = df_ok[['clase','n_heavy','e_xtb_hartree']].dropna()

if len(df_g2) > 3:
    df_g2 = df_g2.copy()
    df_g2['e_por_atomo'] = df_g2['e_xtb_hartree'] / df_g2['n_heavy']

    clases = sorted(df_g2['clase'].unique())
    colors = plt.cm.Set2(np.linspace(0, 1, len(clases)))

    for ax, col, etiqueta in [
        (axes[0], 'e_xtb_hartree', 'Energía total (Hartree)'),
        (axes[1], 'e_por_atomo',   'Energía / átomo pesado (Hartree/átomo)')
    ]:
        datos = [df_g2[df_g2['clase']==c][col].dropna() for c in clases]
        datos_v = [(c, d) for c, d in zip(clases, datos) if len(d) > 0]
        if datos_v:
            cls_v, dat_v = zip(*datos_v)
            bp = ax.boxplot(dat_v, labels=cls_v, patch_artist=True,
                            medianprops={'color':'red','linewidth':2})
            for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
                patch.set_facecolor(color)
            ax.set_ylabel(etiqueta, fontsize=11)
            ax.set_title(etiqueta.split('(')[0].strip(), fontsize=11)
            ax.tick_params(axis='x', rotation=25)

plt.suptitle('Distribución de energías GFN2-xTB por clase estructural', fontsize=12)
plt.tight_layout()
plt.savefig('p01_energia_por_clase.pdf', bbox_inches='tight')
plt.show()

print("Observa la diferencia entre el gráfico izquierdo y el derecho.")
print("¿Cuál es más informativo? ¿Por qué?")
print()
print("Pregunta de reporte: ¿qué clase tiene la energía media más negativa")
print("por átomo? ¿Cómo se relaciona con la estabilización aromática?")


In [ ]:
# ── Análisis de fallos de incrustación ────────────────────────
# Pregunta: ¿qué tienen en común las moléculas donde ETKDG falló?

df_fallo = df[~df['embed_ok']]

print(f"Moléculas con fallo en la incrustación: {len(df_fallo)} / {len(df)}")
print(f"Tasa de éxito global: {df['embed_ok'].mean()*100:.1f}%")
print()

if len(df_fallo) > 0:
    print("Detalle de los fallos:")
    print(df_fallo[['id', 'smiles', 'clase']].to_string(index=False))
    print()
    print("Examina los SMILES de los fallos:")
    print("  ¿Tienen anillos pequeños (< 5 átomos)?")
    print("  ¿Tienen geometría de tensión (biciclobutano, cubano)?")
    print("  ¿Tienen quiralidad axial o planar?")
    print()
    print("Tasas de fallo por clase:")
    fallo_clase = df.groupby('clase')['embed_ok'].agg(['sum','count'])
    fallo_clase['tasa_exito'] = fallo_clase['sum'] / fallo_clase['count'] * 100
    print(fallo_clase.to_string())
else:
    print("Ninguna molécula falló en este dataset.")
    print("(El dataset completo de 50 incluye casos difíciles con fallos.)")


In [ ]:
# ── Mapa de correlaciones ─────────────────────────────────────
# Pregunta: ¿qué variable predice mejor el tiempo de optimización?
# ¿Tiene sentido esa correlación físicamente?

import seaborn as sns

cols_corr = [c for c in ['n_heavy','n_rot_bonds','mw','e_xtb_hartree','t_xtb_s']
             if c in df_ok.columns]
df_corr   = df_ok[cols_corr].dropna()

if len(df_corr) > 5:
    fig, ax = plt.subplots(figsize=(6, 5))
    corr = df_corr.corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, ax=ax, square=True,
                cbar_kws={'shrink': 0.8})
    ax.set_title('Correlaciones entre variables del dataset', fontsize=12)
    plt.tight_layout()
    plt.savefig('p01_correlaciones.pdf', bbox_inches='tight')
    plt.show()

    print("Correlaciones más fuertes (|r| > 0.7):")
    encontradas = False
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            r = corr.iloc[i, j]
            if abs(r) > 0.7:
                print(f"  {corr.columns[i]:15s} ↔ {corr.columns[j]:15s}: r = {r:.3f}")
                encontradas = True
    if not encontradas:
        print("  Ninguna correlación supera |r| = 0.7 en este dataset.")
    print()
    print("¿La variable con mayor correlación con t_xtb_s es la que esperabas?")


---
## 🔄 Revisión de hipótesis iniciales

Vuelve a las respuestas que escribiste al principio.


In [31]:
# ── Revisión de las respuestas previas ───────────────────────
print("═" * 55)
print("  TUS RESPUESTAS PREVIAS")
print("═" * 55)
print()
print("Pregunta 1 (representaciones 0D/1D/2D/3D):")
print(respuesta_1)
print()
print("Pregunta 2 (efecto de la semilla aleatoria):")
print(respuesta_2)
print()
print("Pregunta 3 (por qué tres métodos en secuencia):")
print(respuesta_3)
print()
print("─" * 55)
print("Después de la práctica, responde en la celda siguiente:")
print("  · ¿Cambiarías alguna respuesta? ¿Por qué?")
print("  · ¿Qué fue lo más sorprendente?")
print("  · ¿Qué pregunta nueva te generó la práctica?")


═══════════════════════════════════════════════════════
  TUS RESPUESTAS PREVIAS
═══════════════════════════════════════════════════════

Pregunta 1 (representaciones 0D/1D/2D/3D):

0D: Contiene únicamente los elementos de la molécula
1D: Supongo que tiene conecciones
2D: Da la distribución bidimendional de todos los grupos
3D: Representación espacial


Pregunta 2 (efecto de la semilla aleatoria):

Sí importa, y me dará respuestas diferentes dependiendo de la semilla usada.


Pregunta 3 (por qué tres métodos en secuencia):

ETKDG resuelve...
MMFF94 resuelve...
GFN2-xTB resuelve...


───────────────────────────────────────────────────────
Después de la práctica, responde en la celda siguiente:
  · ¿Cambiarías alguna respuesta? ¿Por qué?
  · ¿Qué fue lo más sorprendente?
  · ¿Qué pregunta nueva te generó la práctica?


In [ ]:
# ── Escribe tu reflexión final ────────────────────────────────
reflexion_final = """
¿Cambiarías alguna respuesta previa?:

¿Qué fue lo más sorprendente de los resultados?:

¿Qué pregunta nueva te generó la práctica?:
"""
print(reflexion_final)


---
## 📦 Entregables

| Etiqueta | Archivo | Descripción |
|:--|:--|:--|
| D1 | `cafeina_GFN2.xyz` | Geometría optimizada de la semilla |
| D2 | `cafeina_xtb.out` | Log completo de xTB sin modificar |
| D3 | `p01_dataset_final.csv` | Dataset bosque con semilla integrada |
| F1 | `p01_tiempo_vs_natom.pdf` | Dispersión tiempo vs átomos pesados |
| F2 | `p01_energia_por_clase.pdf` | Boxplot energía por clase |
| F3 | `p01_correlaciones.pdf` | Mapa de calor de correlaciones |
| T1 | Reporte (≤ 2 páginas) | Ver especificaciones abajo |

**Especificaciones del reporte T1:**
- Tabla de validación geométrica vs cristal (completa con tus valores calculados)
- F1 y F2 comentadas con interpretación química (no solo descripción)
- Respuesta a las 3 preguntas de discusión (≤ 150 palabras cada una)
- Una reflexión sobre las limitaciones del protocolo ETKDG + MMFF94 + GFN2-xTB


In [ ]:
# ── Checklist automático de entregables ──────────────────────
import pathlib

entregables = {
    'cafeina_GFN2.xyz'          : 'D1 — Geometría semilla GFN2-xTB',
    'cafeina_xtb.out'           : 'D2 — Log de xTB',
    'p01_dataset_final.csv'     : 'D3 — Dataset final (50 + semilla)',
    'p01_tiempo_vs_natom.pdf'   : 'F1 — Tiempo vs átomos',
    'p01_energia_por_clase.pdf' : 'F2 — Energía por clase',
    'p01_correlaciones.pdf'     : 'F3 — Mapa de correlaciones',
}

print("═" * 55)
print("  CHECKLIST DE ENTREGABLES")
print("═" * 55)
todos = True
for archivo, desc in entregables.items():
    existe = pathlib.Path(archivo).exists()
    if existe:
        size = pathlib.Path(archivo).stat().st_size
        print(f"  ✓  {desc:38s} ({size:,} bytes)")
    else:
        print(f"  ✗  {desc:38s} ← FALTA")
        todos = False

print()
if todos:
    print("✓ Todos los entregables están listos para entregar.")
else:
    print("⚠ Algunos entregables faltan. Revisa las celdas marcadas con ✗.")
    print("  Consulta el número de celda en los títulos de sección.")


---
## ❓ Preguntas de discusión

Responde en el reporte escrito (≤ 150 palabras por pregunta).
Puedes usar esta celda para esbozar ideas.


In [ ]:
preguntas = {
    1: """(Comprensión) El SMILES de la cafeína es Cn1cnc2c1c(=O)n(c(=O)n2C)C.
Escribe el SMILES de la teobromina (7-metilxantina) y la teofilina (1,3-dimetilxantina).
¿Qué diferencia estructural codifica cada SMILES respecto al de la cafeína?""",

    2: """(Comprensión) ¿Qué significan --chrg 0 y --uhf 0 en el comando xtb?
¿Qué ocurriría química y numéricamente si ejecutaras el cálculo
con --uhf 2 para la cafeína?""",

    3: """(Análisis de log) Del archivo cafeina_xtb.out, extrae e interpreta:
(a) número total de ciclos de optimización,
(b) energía en el ciclo 1 vs el último (¿cuánto bajó en kcal/mol?),
(c) gradiente RMS final vs el criterio --opt tight.
¿Fue útil la pre-optimización con MMFF94?""",

    4: """(Metodológico) La energía GFN2-xTB de la cafeína es un número negativo
en Hartree. ¿Por qué la energía de las moléculas más grandes del bosque
es más negativa todavía? ¿Puedes usar la energía absoluta para comparar
la estabilidad de dos moléculas de diferente tamaño? ¿Qué deberías hacer?""",

    5: """(Crítico) En el bosque, algunas moléculas fallan en la incrustación ETKDG.
¿Qué tienen en común estructuralmente? Propón una estrategia alternativa
para generar geometrías iniciales para esas moléculas problemáticas.""",
}

for n, q in preguntas.items():
    print(f"Pregunta {n}:")
    print(q)
    print()


---
## 🔗 Conexión con el resto del manual

```{admonition} Guarda estos archivos — los usarás en prácticas futuras
:class: tip

- `cafeina_GFN2.xyz` → input directo de la **Práctica 4**
  (primera optimización DFT B3LYP/6-31G(d))
- `p01_dataset_final.csv` → base del análisis conformacional
  de la **Práctica 3**
- El pipeline SMILES → 3D de esta práctica se reutiliza
  en **todas** las prácticas de los Bloques 17–20
```

| Habilidad aprendida | Herramienta | Vuelve a aparecer en... |
|:--|:--|:--|
| SMILES → grafo molecular | `Chem.MolFromSmiles` | P02, P03, P47–P50 |
| Grafo → coordenadas 3D | `AllChem.ETKDGv3` | P02, P03 |
| Optimización FF | MMFF94 | P03 |
| Optimización semiempírica | `xtb --opt tight` | P03, P10, P34 |
| Leer logs con regex | `re.search` | P04–P18 |
| RMSD geométrico | NumPy + SVD | P10, P43–P45 |
| Dataset estructural | pandas | P47–P55 |
